## Hill Climbing

In [1]:
import os
import glob
import re
import pandas as pd
import numpy as np
from sklearn.metrics import balanced_accuracy_score
import torch
import seaborn as sb
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
 
# ============================================================
# Configuration
# ============================================================

OOF_PATH = "solution_metadata/oof"
OOF_FILENAME_PATTERN = r"^(.*)-([0-9]+\.[0-9]+)-oof\.csv$"
COL_ALIGNMENT = ["id", "at-risk", "unhealthy", "fit"]

TARGET_MAPPING = {
    "at-risk": 0,
    "unhealthy": 1,
    "fit": 2
}

prob_cols = list(TARGET_MAPPING.keys())

y = (
    pd.read_csv("input_data/playground-series-s6e7/train.csv")["health_condition"]
    .map(TARGET_MAPPING)
    .values
)

PROB_PATH = "solution_metadata/prob"
PROB_FILENAME_PATTERN = r"^(.*)-([0-9]+\.[0-9]+)-t_prob\.csv$"

COL_ALIGNMENT = ["id", "at-risk", "unhealthy", "fit"]

TARGET_MAPPING = {
    "at-risk": 0,
    "unhealthy": 1,
    "fit": 2
}

prob_cols = list(TARGET_MAPPING.keys())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"device: {device} ...")

device: cpu ...


In [2]:
preds = {}
scores = {}

for file in sorted(glob.glob(os.path.join(OOF_PATH, "*.csv"))):
    filename = os.path.basename(file)
    match = re.match(OOF_FILENAME_PATTERN, filename)
    if match is None:
        print(f"Skipping {filename}")
        continue

    score = float(match.group(2))
    model_name = filename.replace("-oof.csv", "")

    print(f"Loading {model_name}")
    df = pd.read_csv(file)
    try:
        df = df[COL_ALIGNMENT]
    except Exception as e:
        print(e)

    # assert df.columns.tolist() == COL_ALIGNMENT, \
    #     f"Column alignment mismatch in {filename}"

    preds[model_name] = df.to_numpy(dtype=np.float32)
    scores[model_name] = score
    print(f"Loaded {model_name:<60} Score = {score:.6f}")

print(f"\nLoaded {len(preds)} OOF files.")


# ============================================================
# Alignment detector 
# ============================================================
def alignment_detector(sub: pd.DataFrame, reference: pd.DataFrame, plot_name: str = None, show_heatmap:bool=False):
    if sub.shape != reference.shape:
        raise ValueError(
            f"Shape mismatch: {sub.shape} vs {reference.shape}"
        )

    n_cols = sub.shape[1]
    cost_matrix = np.empty((n_cols, n_cols))

    for i in range(n_cols):
        sub_col = sub.iloc[:, i].to_numpy()
        for j in range(n_cols):
            ref_col = reference.iloc[:, j].to_numpy()
            cost_matrix[i, j] = np.abs(sub_col - ref_col).sum()

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    mapping = dict(zip(row_ind, col_ind))
    aligned = pd.DataFrame(index=sub.index)
    for sub_idx, ref_idx in mapping.items():
        aligned[reference.columns[ref_idx]] = sub.iloc[:, sub_idx].values

    aligned = aligned[reference.columns]

    if show_heatmap:
        diff = np.abs(aligned - reference)

        plt.figure(figsize=(6, 2))
        sb.heatmap(diff, cmap="viridis", cbar=True)

        title = "Alignment Check"
        if plot_name is not None:
            title += f" ({plot_name})"

        plt.title(title)
        plt.xlabel("Columns")
        plt.ylabel("Rows")
        plt.show()

    print("\nColumn Mapping")
    print("-" * 50)

    for sub_idx, ref_idx in sorted(mapping.items()):
        print(
            f"{sub.columns[sub_idx]:<20}"
            f" --> "
            f"{reference.columns[ref_idx]}"
        )

    total_cost = cost_matrix[row_ind, col_ind].sum()
    print(f"\nTotal alignment cost : {total_cost:.6f}")

    return aligned


preds_corrected = {}
reference = pd.DataFrame(preds["rmlp-v1-0.95062"], columns=COL_ALIGNMENT)
for idx, (model_name, pred) in enumerate(preds.items()):
    df = pd.DataFrame(pred, columns=COL_ALIGNMENT)
    preds_corrected[model_name] = alignment_detector(df, reference, plot_name=model_name, show_heatmap=False)

Loading cat-v1-0.949450
Loaded cat-v1-0.949450                                              Score = 0.949450
Loading cat-v1-0.949475
Loaded cat-v1-0.949475                                              Score = 0.949475
Loading cat-v1-0.949507
Loaded cat-v1-0.949507                                              Score = 0.949507
Loading cat-v2-0.949571
Loaded cat-v2-0.949571                                              Score = 0.949571
Loading cat-v2-0.949629
Loaded cat-v2-0.949629                                              Score = 0.949629
Loading cat-v2-0.949934
"['at-risk', 'unhealthy', 'fit'] not in index"
Loaded cat-v2-0.949934                                              Score = 0.949934
Loading ft-v1-0.95029
Loaded ft-v1-0.95029                                                Score = 0.950290
Loading hgbc-v1-0.95026
Loaded hgbc-v1-0.95026                                              Score = 0.950260
Loading hgbc-v1-0.95034
Loaded hgbc-v1-0.95034                                     

In [3]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score

preds = {
    name: df.to_numpy(dtype=np.float32)
    for name, df in preds_corrected.items()
}

sorted_models = sorted(
    scores.items(),
    key=lambda x: x[1],
    reverse=True
)

best_sol = sorted_models[0][0]
best_score = sorted_models[0][1]
best_score = 0.95050

print(f"Starting Model : {best_sol}")
print(f"Score          : {best_score:.6f}")


weights_pool = np.linspace(0.01, 0.99, 99)
running_ensemble = preds[best_sol].copy()

best_cv = best_score
selected_models = [best_sol]
final_weights = {best_sol: 1.0}

for model_name, _ in sorted_models:
    if model_name == best_sol:
        continue

    pred = preds[model_name]

    local_best_score = best_cv
    local_best_weight = None
    local_best_pred = None

    for w in weights_pool:

        trial = (1 - w) * running_ensemble + w * pred
        score = balanced_accuracy_score(y, np.argmax(trial, axis=1))

        if score > local_best_score:
            local_best_score = score
            local_best_weight = w
            local_best_pred = trial

    if local_best_weight is None:
        print(f"[SKIP] {model_name}")
        continue

    print(
        f"[ADD ] {model_name} "
        f"weight={local_best_weight:.2f} "
        f"{best_cv:.6f} -> {local_best_score:.6f}"
    )

    running_ensemble = local_best_pred
    best_cv = local_best_score
    selected_models.append(model_name)

    for k in final_weights:
        final_weights[k] *= (1 - local_best_weight)

    final_weights[model_name] = local_best_weight

# ============================================================
# Normalize (floating-point safety)
# ============================================================

total = sum(final_weights.values())

for k in final_weights:
    final_weights[k] /= total

reconstructed = np.zeros_like(preds[best_sol], dtype=np.float32)
for model, weight in final_weights.items():
    reconstructed += weight * preds[model]

running_pred = np.argmax(running_ensemble, axis=1)
reconstructed_pred = np.argmax(reconstructed, axis=1)
running_score = balanced_accuracy_score(y, running_pred)

reconstructed_score = balanced_accuracy_score(y, reconstructed_pred)

print()

print("Running Score      :", running_score)
print("Reconstructed Score:", reconstructed_score)

print("Prediction Equal :", np.array_equal(running_pred, reconstructed_pred))
print("Probability Equal:", np.allclose(running_ensemble, reconstructed, atol=1e-8))
print("FINAL_WEIGHTS = {")

for model, weight in sorted(final_weights.items(), key=lambda x: x[1], reverse=True):
    print(f'    "{model}": {weight:.12f},')
print("}")


# test_preds = {k: v.to_numpy(dtype=np.float32) for k, v in test_preds_corrected.items()}
# test_ensemble = np.zeros_like(next(iter(test_preds.values())), dtype=np.float32)

# for model, weight in FINAL_WEIGHTS.items():
#     test_ensemble += weight * test_preds[model]
# submission[target_cols] = test_ensemble

Starting Model : rmlp-v3-0.95065
Score          : 0.950500
[SKIP] rmlp-v2-0.95063
[SKIP] rmlp-v1-0.95062
[SKIP] rlmp-v1-0.95048
[SKIP] hgbc-v1-0.95034
[SKIP] ft-v1-0.95029
[SKIP] hgbc-v1-0.95026
[SKIP] xgb-v1-0.95014
[SKIP] xgb-v1-0.95005
[SKIP] cat-v2-0.949934
[SKIP] lgb-v1-0.949669
[SKIP] cat-v2-0.949629
[SKIP] cat-v2-0.949571
[SKIP] lgb-v1-0.94954
[SKIP] cat-v1-0.949507
[SKIP] cat-v1-0.949475


KeyboardInterrupt: 

## Torch

In [ ]:
import torch
import time
from sklearn.metrics import balanced_accuracy_score

print(f"Using Pytorch & Device : {device}")
start = time.perf_counter()

preds = {
    name: torch.tensor(
        df.to_numpy(dtype=np.float32),
        dtype=torch.float32,
        device=device,
    )
    for name, df in preds_corrected.items()
}

sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)

best_sol = sorted_models[0][0]
best_cv = 0.95050

weights_pool = torch.linspace(
    0.01,
    0.99,
    99,
    device=device,
)

w_grid = weights_pool.view(-1, 1, 1)

running_ensemble = preds[best_sol].clone()

selected_models = [best_sol]
final_weights = {best_sol: 1.0}

y_np = np.asarray(y)

print(f"Starting Model : {best_sol}")
print(f"Score          : {best_cv:.6f}")

for model_name, _ in sorted_models:

    if model_name == best_sol:
        continue

    pred = preds[model_name]

    trials = (
        (1 - w_grid) * running_ensemble.unsqueeze(0)
        + w_grid * pred.unsqueeze(0)
    )

    trial_preds = torch.argmax(trials, dim=2)

    local_best_score = best_cv
    local_best_weight = None
    local_best_idx = None

    for idx in range(len(weights_pool)):

        score = balanced_accuracy_score(
            y_np,
            trial_preds[idx].cpu().numpy()
        )

        if score > local_best_score:
            local_best_score = score
            local_best_weight = weights_pool[idx].item()
            local_best_idx = idx

    if local_best_weight is None:
        print(f"[SKIP] {model_name}")
        continue

    print(
        f"[ADD ] {model_name} "
        f"weight={local_best_weight:.2f} "
        f"{best_cv:.6f} -> {local_best_score:.6f}"
    )

    running_ensemble = trials[local_best_idx]
    best_cv = local_best_score

    selected_models.append(model_name)

    for k in final_weights:
        final_weights[k] *= (1 - local_best_weight)

    final_weights[model_name] = local_best_weight

total = sum(final_weights.values())

for k in final_weights:
    final_weights[k] /= total

reconstructed = torch.zeros_like(preds[best_sol])

for model, weight in final_weights.items():
    reconstructed += weight * preds[model]

running_pred = torch.argmax(running_ensemble, dim=1).cpu().numpy()
reconstructed_pred = torch.argmax(reconstructed, dim=1).cpu().numpy()

print("Running Score      :", balanced_accuracy_score(y, running_pred))
print("Reconstructed Score:", balanced_accuracy_score(y, reconstructed_pred))
print("Prediction Equal :", np.array_equal(running_pred, reconstructed_pred))
print(
    "Probability Equal:",
    torch.allclose(running_ensemble, reconstructed, atol=1e-8),
)

end = time.perf_counter()

execution_time = end - start
print(f"Execution time: {execution_time:.4f} seconds")

Using Pytorch & Device : cpu
Starting Model : rmlp-v3-0.95065
Score          : 0.950500


## Numpy

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score

start = time.perf_counter()

preds = {
    name: df.to_numpy(dtype=np.float32)
    for name, df in preds_corrected.items()
}

sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)

best_sol = sorted_models[0][0]
best_cv = 0.95050

weights_pool = np.linspace(0.01, 0.99, 99, dtype=np.float32)
w_grid = weights_pool[:, None, None]

running_ensemble = preds[best_sol].copy()

selected_models = [best_sol]
final_weights = {best_sol: 1.0}

print(f"Starting Model : {best_sol}")
print(f"Score          : {best_cv:.6f}")

for model_name, _ in sorted_models:

    if model_name == best_sol:
        continue

    pred = preds[model_name]

    trials = (
        (1 - w_grid) * running_ensemble[None]
        + w_grid * pred[None]
    )

    trial_preds = np.argmax(trials, axis=2)

    local_best_score = best_cv
    local_best_weight = None
    local_best_idx = None

    for idx in range(len(weights_pool)):

        score = balanced_accuracy_score(y, trial_preds[idx])

        if score > local_best_score:
            local_best_score = score
            local_best_weight = weights_pool[idx]
            local_best_idx = idx

    if local_best_weight is None:
        print(f"[SKIP] {model_name}")
        continue

    print(
        f"[ADD ] {model_name} "
        f"weight={local_best_weight:.2f} "
        f"{best_cv:.6f} -> {local_best_score:.6f}"
    )

    running_ensemble = trials[local_best_idx]
    best_cv = local_best_score

    selected_models.append(model_name)

    for k in final_weights:
        final_weights[k] *= (1 - local_best_weight)

    final_weights[model_name] = float(local_best_weight)

total = sum(final_weights.values())

for k in final_weights:
    final_weights[k] /= total

reconstructed = np.zeros_like(preds[best_sol], dtype=np.float32)

for model, weight in final_weights.items():
    reconstructed += weight * preds[model]

running_pred = np.argmax(running_ensemble, axis=1)
reconstructed_pred = np.argmax(reconstructed, axis=1)

print("Running Score      :", balanced_accuracy_score(y, running_pred))
print("Reconstructed Score:", balanced_accuracy_score(y, reconstructed_pred))
print("Prediction Equal :", np.array_equal(running_pred, reconstructed_pred))
print("Probability Equal:", np.allclose(running_ensemble, reconstructed, atol=1e-8))

end = time.perf_counter()

execution_time = end - start
print(f"Execution time: {execution_time:.4f} seconds")

In [ ]:
# ============================================================
# Load Test Probabilities
# ============================================================

prob_data = {}

for file in sorted(glob.glob(os.path.join(PROB_PATH, "*.csv"))):
    filename = os.path.basename(file)
    match = re.match(PROB_FILENAME_PATTERN, filename)

    if match is None:
        print(f"Skipping {filename}")
        continue

    # Use filename without "-t_prob.csv" as model name
    model_name = filename.replace("-t_prob.csv", "")
    df = pd.read_csv(file)[COL_ALIGNMENT]
    assert df.columns.tolist() == COL_ALIGNMENT, \
        f"Column alignment mismatch in {filename}"
    prob_data[model_name] = df
    print(f"Loaded {model_name}")

# ============================================================
# Hill Climbing Blend
# ============================================================

base_model = hc_ensembles[0]

blend = pd.DataFrame()
blend["id"] = prob_data[base_model]["id"]

ensemble = prob_data[base_model][prob_cols].copy()

for model_name, weight in hc_weights:
    ensemble = (
        (1.0 - float(weight)) * ensemble + float(weight) * prob_data[model_name][prob_cols]
    )

blend[prob_cols] = ensemble
blend["class"] = blend[prob_cols].idxmax(axis=1)

submission = blend[["id", "class"]]

submission.to_csv(
    "solution_metadata/blends/hill_climbing_blend_v15.csv",
    index=False
)

print("\n==================================================")
print("Submission Saved")
print("==================================================")
print(f"Output file : solution_metadata/blends/hill_climbing_blend_v15.csv")
print(f"Models used : {len(hc_ensembles)}")

print("\nBase Model")
print("----------")
print(base_model)

print("\nHill Climbing Order")
print("-------------------")
for i, model in enumerate(hc_ensembles, 1):
    print(f"{i}. {model}")

print("\nWeights")
print("-------")
for model, weight in hc_weights:
    print(f"{model:<60} {float(weight):.2f}")

Loaded cat-v1-0.949450
Loaded cat-v1-0.949475
Loaded cat-v1-0.949507
Loaded cat-v2-0.949571
Loaded cat-v2-0.949629
Loaded ft-v1-0.95029
Loaded hgbc-v1-0.95026
Loaded hgbc-v1-0.95034
Loaded lgb-v1-0.949366
Loaded lgb-v1-0.94954
Loaded lgb-v1-0.949669
Loaded rlmp-v1-0.95048
Loaded rmlp-v1-0.95062
Loaded rmlp-v2-0.95063
Loaded rmlp-v3-0.95065
Loaded xgb-v1-0.95005
Loaded xgb-v1-0.95014

Submission Saved
Output file : solution_metadata/blends/hill_climbing_blend_v14.csv
Models used : 5

Base Model
----------
rmlp-v3-0.95065

Hill Climbing Order
-------------------
1. rmlp-v3-0.95065
2. rmlp-v1-0.95062
3. hgbc-v1-0.95034
4. ft-v1-0.95029
5. cat-v2-0.949629

Weights
-------
rmlp-v1-0.95062                                              0.30
hgbc-v1-0.95034                                              0.05
ft-v1-0.95029                                                0.12
cat-v2-0.949629                                              0.02


## Meta Stacking

In [ ]:
import copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold 

# ============================================================
# Meta stacking with an Optimized PyTorch Tabular MLP
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class TabularMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        # Switched to LayerNorm since input features are concatenated probabilities [0, 1]
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.30),  # Slightly higher dropout to prevent meta-overfitting
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.network(x)

model_names = sorted(preds.keys())
if not model_names:
    raise RuntimeError("No OOF predictions were loaded; cannot build a meta stack.")

if not all(name in prob_data for name in model_names):
    missing = [name for name in model_names if name not in prob_data]
    raise RuntimeError(f"Missing test probabilities for: {missing}")


def probability_to_logits(prob_array, epsilon=1e-7):
    """
    Stabilizes and transforms probabilities into log-odds (logits).
    Works for both binary and multiclass probability matrices.
    """
    # Clip to avoid log(0) or division by zero errors
    prob_clipped = np.clip(prob_array, epsilon, 1.0 - epsilon)
    return np.log(prob_clipped / (1.0 - prob_clipped))

# Concatenate and transform OOF predictions
X_train_list = []
for name in model_names:
    raw_probs = preds[name].astype(np.float32)
    logit_features = probability_to_logits(raw_probs)
    X_train_list.append(logit_features)

X_train = np.concatenate(X_train_list, axis=1)

# Concatenate and transform Test predictions
X_test_list = []
for name in model_names:
    raw_test_probs = prob_data[name][prob_cols].to_numpy(dtype=np.float32)
    logit_test_features = probability_to_logits(raw_test_probs)
    X_test_list.append(logit_test_features)

X_test = np.concatenate(X_test_list, axis=1)


# Train a 5-fold CV meta-model.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_meta = np.zeros((len(y), 3), dtype=np.float32)

# Array to collect test predictions from each fold model
test_preds_folds = np.zeros((len(X_test), 3), dtype=np.float32)

EPOCHS = 35  # Increased epochs to give the network time to converge
BATCH_SIZE = 1024  # Reduced batch size for better gradient granularity
VERBOSE = 2

X_test_t = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y)):
    print(f"\n--- Training Fold {fold + 1} ---")
    X_tr = torch.tensor(X_train[tr_idx], dtype=torch.float32, device=DEVICE)
    y_tr = torch.tensor(y[tr_idx], dtype=torch.long, device=DEVICE)
    X_va = torch.tensor(X_train[va_idx], dtype=torch.float32, device=DEVICE)
    y_va = torch.tensor(y[va_idx], dtype=torch.long, device=DEVICE)

    counts = torch.bincount(y_tr)
    class_weights = counts.sum() / (len(counts) * counts.float())
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    model = TabularMLP(input_dim=X_tr.shape[1], num_classes=3).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    train_ds = TensorDataset(X_tr, y_tr)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    best_score = -1.0
    best_state = None

    for epoch in range(EPOCHS):
        model.train()
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_pred = torch.argmax(val_logits, dim=1)
            score = balanced_accuracy_score(y_va.cpu().numpy(), val_pred.cpu().numpy())

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
        
        if epoch % VERBOSE == 0 or epoch == EPOCHS - 1:
            print(f"Epoch {epoch:4d} | Balanced Accuracy: {score:.5f}")

    print(f"Fold {fold + 1} Best Balanced Accuracy: {best_score:.5f}")
    
    # Load best checkpoint for OOF and Test Inference
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        oof_meta[va_idx] = model(X_va).softmax(dim=1).cpu().numpy()
        # Accumulate test predictions (Blends predictions across all fold models)
        test_preds_folds += model(X_test_t).softmax(dim=1).cpu().numpy() / cv.n_splits

# Final CV score tracking
meta_score = balanced_accuracy_score(y, np.argmax(oof_meta, axis=1))
print(f"\nMeta stacking CV balanced accuracy: {meta_score:.6f}")

# Output Processing
output_dir = Path("solution_metadata/meta_stack")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "meta_mlp_stack_v6.csv"

blend = pd.DataFrame({"id": prob_data[model_names[0]]["id"]})
blend[prob_cols] = test_preds_folds
blend["class"] = blend[prob_cols].idxmax(axis=1)
blend[["id", "class"]].to_csv(output_path, index=False)

print("\n==================================================")
print("Meta Stacking Submission Generated")
print("==================================================")
print(f"CV balanced accuracy : {meta_score:.6f}")
print(f"Saved submission     : {output_path}")
print(f"Base models used     : {len(model_names)}")

Using device: cpu

--- Training Fold 1 ---
Epoch    0 | Balanced Accuracy: 0.95057
Epoch    2 | Balanced Accuracy: 0.95136
Epoch    4 | Balanced Accuracy: 0.95162
Epoch    6 | Balanced Accuracy: 0.95045


KeyboardInterrupt: 